In [1]:
from gsloc.inference.test import TestConfig, Test
from pathlib import Path
from gsloc.models import opr_graph_extention as network 
import torch
from torchvision.transforms import functional as F
from mmpr.models import MegaLoc
from gsloc.datasets import ThreeRScan

from torchvision import transforms as T
from gsloc.utils.visual import plot_metrics_from_parquet, plot_metrics_from_experiment_dir

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


2026-05-07 06:36:04.633 | WARNING  | opr.optional_deps:warn_once:115 - MinkowskiEngine is not available. sparse convolutions will be disabled. See the documentation for installation instructions


In [2]:
# weights_path = Path("/mnt/external_usb_hdd/6YL/Datasets/3RScan/weights/graphs/gatv1/best_model.pth")
# ckpt = torch.load(weights_path, map_location="cpu", weights_only=False)
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# OPR_GAT_graph_encoder = network.OPR_GATGraphEncoder(
#     in_dim=4,
#     hidden_dim=512,
#     n_layers=1,
#     num_node_classes=529, 
#     node_emb_dim=128,
#     num_edge_classes=41,
#     edge_emb_dim=128,
#     proj_dim=256,
#     edge_cont_dim=10,
#     dropout=0.1,
#     heads=4
#     ).to(device)
    
# # megaloc = torch.hub.load("gmberton/MegaLoc", "get_trained_model")
# # image_encoder = megaloc.to(device)

# graph_model1 = network.OPR_MultiModalVPRGraphEncoder(
#     graph_encoder=OPR_GAT_graph_encoder,
#     image_encoder=None,
#     image_out_dim=8448,
#     graph_out_dim=256,
#     fusion_dim=8448,
#     normalize=True,
#     graph_fusion_scale=0.05,
#     freeze_image_encoder=True,
#     mode="graph")

# missing, unexpected = graph_model1.load_state_dict(ckpt["model_state_dict"], strict=False)
# ignored_unexpected_prefixes = ("image_encoder.", "graph_encoder.convs.")
# unexpected_other = [k for k in unexpected if not k.startswith(ignored_unexpected_prefixes)]
# if unexpected_other:
#     raise RuntimeError(f"Unexpected checkpoint keys: {unexpected_other}")
# # ``missing`` includes MegaLoc hub weights and GINE conv params; those ckpt tensors appear under ``ignored_unexpected_prefixes``.

# graph_model1.to(device)
# graph_model1.eval()

In [3]:
weights_path = Path("/mnt/external_usb_hdd/6YL/Datasets/3RScan/weights/graphs/gatV3/best_model.pth")
ckpt = torch.load(weights_path, map_location="cpu", weights_only=False)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

GAT_graph_encoder = network.OPR_GATGraphEncoder64(
    in_dim=4,
    hidden_dim=512,
    n_layers=1,
    num_node_classes=529, 
    node_emb_dim=64,
    num_edge_classes=41,
    edge_emb_dim=128,
    proj_dim=64,
    edge_cont_dim=10,
    dropout=0.1,
    heads=4).to(device)

graph_model2 = network.OPR_GraphEnhancedMegaloc64(
    graph_encoder=GAT_graph_encoder,
    image_encoder=None
)

missing, unexpected = graph_model2.load_state_dict(ckpt["multimodal_state_dict"], strict=False)
ignored_unexpected_prefixes = ("image_encoder.", "graph_encoder.convs.")
unexpected_other = [k for k in unexpected if not k.startswith(ignored_unexpected_prefixes)]
if unexpected_other:
    raise RuntimeError(f"Unexpected checkpoint keys: {unexpected_other}")
# ``missing`` includes MegaLoc hub weights and GINE conv params; those ckpt tensors appear under ``ignored_unexpected_prefixes``.

graph_model2.to(device)
graph_model2.eval()

OPR_GraphEnhancedMegaloc64(
  (graph_encoder): OPR_GATGraphEncoder64(
    (edge_cont_ln): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
    (edge_lbl_ln): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
    (node_emb): Embedding(529, 64)
    (edge_emb): Embedding(41, 128)
    (edge_cont_mlp): Sequential(
      (0): Linear(in_features=10, out_features=512, bias=True)
      (1): ReLU(inplace=True)
      (2): Linear(in_features=512, out_features=512, bias=True)
    )
    (edge_gate): Sequential(
      (0): Linear(in_features=1024, out_features=512, bias=True)
      (1): ReLU(inplace=True)
      (2): Linear(in_features=512, out_features=512, bias=True)
      (3): Sigmoid()
    )
    (edge_label_proj): Sequential(
      (0): Linear(in_features=128, out_features=512, bias=True)
      (1): ReLU(inplace=True)
      (2): Linear(in_features=512, out_features=512, bias=True)
    )
    (edge_fuse): Sequential(
      (0): Linear(in_features=1024, out_features=512, bias=True)
      (1)

In [4]:
megaLoc = MegaLoc()
megaLoc.to(device)
megaLoc.eval()

Using cache found in /home/kartashov_ga/.cache/torch/hub/gmberton_MegaLoc_main
Using cache found in /home/kartashov_ga/.cache/torch/hub/facebookresearch_dinov2_main
/home/kartashov_ga/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/home/kartashov_ga/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/home/kartashov_ga/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


MegaLoc(
  (model): MegaLocModel(
    (backbone): DINOv2(
      (model): DinoVisionTransformer(
        (patch_embed): PatchEmbed(
          (proj): Conv2d(3, 768, kernel_size=(14, 14), stride=(14, 14))
          (norm): Identity()
        )
        (blocks): ModuleList(
          (0-11): 12 x NestedTensorBlock(
            (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
            (attn): MemEffAttention(
              (qkv): Linear(in_features=768, out_features=2304, bias=True)
              (proj): Linear(in_features=768, out_features=768, bias=True)
              (proj_drop): Dropout(p=0.0, inplace=False)
            )
            (ls1): LayerScale()
            (drop_path1): Identity()
            (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
            (mlp): Mlp(
              (fc1): Linear(in_features=768, out_features=3072, bias=True)
              (act): GELU(approximate='none')
              (fc2): Linear(in_features=3072, out_features=768, 

In [5]:
dataset_path = Path("/mnt/external_usb_hdd/6YL/Datasets/3RScan")
test_dir = Path("/home/kartashov_ga/projects/GSLoc/data/tests/26-05-06/makarov/64xMegaloc")
index_path = "/home/kartashov_ga/projects/GSLoc/data/tests/26-05-06/makarov/64_graph_index"
rerank_index_path = "/home/kartashov_ga/projects/GSLoc/data/tests/26-05-06/megaloc_index"
query_cache_path = "/home/kartashov_ga/projects/GSLoc/data/tests/26-05-06/makarov/64_graph_query_cache"
rerank_query_cache_path = "/home/kartashov_ga/projects/GSLoc/data/tests/26-05-06/megaloc_query_cache"

similarity_names = ["room-sim", "pose-far-sim", "pose-near-sim"]
filter_names = ["base_seq_report", "nearfilter_seq_report", "farfilter_seq_report"]

graph_path = "SceneGraphs_Makarov_FULL_TEST_pt"
edge_normalizer_path="/mnt/external_usb_hdd/6YL/Datasets/3RScan/weights/graphs/gatV3/edge_normalizer (1).pt"
scene_list_path = "/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/test_resplit_scans.txt"
room_json_path = "/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/3RScan.json"

bench_report_dir = test_dir / similarity_names[1]
frames_path = Path("/home/kartashov_ga/projects/GSLoc/data/tests/26-05-06/makarov/64_pure_graph/frames.npz")

seq_filter_kwargs_list = [{
    "seq_similarity_filter_mode": "none",
},
{
    "seq_similarity_filter_mode": "pose",
    "seq_similarity_trans_tol_m": 0.5,
    "seq_similarity_rot_tol_deg": 15
},{
    "seq_similarity_filter_mode": "pose",
    "seq_similarity_trans_tol_m": 1,
    "seq_similarity_rot_tol_deg": 30
},
]
models = [graph_model2]
rerank_models = [megaLoc]

similarity_kwargs_list = [
    {
        "mode": "room",
    },
    {
        "mode": "pose",
        "trans_tol_m": 3,
        "rot_tol_deg": 180
    },
    {
        "mode": "pose",
        "trans_tol_m": 2,
        "rot_tol_deg": 90
    },
]


image_transform_fn = T.Compose([
    T.ToTensor(),
    T.Lambda(lambda x: F.rotate(x, angle=-90)),  # 90° clockwise
    T.Normalize(mean=[0.44420420130352495, 0.41322746532289134, 0.3678658064565412], std=[0.24352604373543688, 0.24045797651069503, 0.24250136992133814]),
    T.Resize([322, 322], antialias=True)
])

cfg = TestConfig(
    dataset_path=dataset_path,
    test_path=test_dir,
    index_path=index_path,
    rerank_index_path=rerank_index_path,
    query_cache_path=query_cache_path,
    rerank_query_cache_path=rerank_query_cache_path,
    bench_report_path=bench_report_dir,
    graph_path=graph_path,   
    dataset_class=ThreeRScan,
    filter_kwargs={"similarity_filter_mode": "none", "similarity_trans_tol_m": 2, "similarity_rot_tol_deg": 90},
    seq_filter_kwargs=seq_filter_kwargs_list[1],
    scene_list_path=scene_list_path,
    room_json_path=room_json_path,
    edge_normalizer_path=edge_normalizer_path,
    image_transform_fn=image_transform_fn,
    graph_feat_dim=4,
    graph_edge_attr_dim=10,
    graph_rotate=True,
    device=device,
    batch_size=16,
    num_workers=4,
    model=graph_model2,
    # rerank_model=megaLoc,
    rerank_k=500,
    per_frame_k_used=25,
    final_k=25,
    seq_lengths=[1, 2, 3, 5, 7, 10, 15, 20, 25, 30, 35],
    recall_at_k=[1, 5, 10, 25],
    similarity_kwargs=similarity_kwargs_list[0],
    std_mode="global",
    scene_df_field="scene",
    pose_df_field="pose",
    frames_path=frames_path
)

In [6]:
test = Test(cfg)
test.run()

2026-05-07 06:36:08.465 | INFO     | gsloc.datasets.three_rscan:__init__:353 - Metadata not found, rebuilding metadata for 3rscan dataset
2026-05-07 06:36:08.490 | INFO     | gsloc.datasets.three_rscan:build_3rscan_df:210 - Scanning 3rscan dataset for 93 selected scenes...


2026-05-07 06:36:20.665 | INFO     | gsloc.datasets.three_rscan:build_3rscan_df:288 - Scanned 21013 rows
2026-05-07 06:36:20.670 | INFO     | mmpr.inference.index:generate:439 - Using existing meta.parquet
100%|██████████| 591/591 [01:10<00:00,  8.44it/s]
2026-05-07 06:37:30.687 | INFO     | mmpr.inference.index:generate:466 - descriptors.npy file was saved in /home/kartashov_ga/projects/GSLoc/data/tests/26-05-06/makarov/64_graph_index
2026-05-07 06:37:30.689 | INFO     | mmpr.inference.index:generate:486 - schema.json file was saved in /home/kartashov_ga/projects/GSLoc/data/tests/26-05-06/makarov/64_graph_index


Running inference and saving frames to /home/kartashov_ga/projects/GSLoc/data/tests/26-05-06/makarov/64_pure_graph/frames.npz


Compute descriptors + PR cache:  31%|███       | 404/1314 [00:52<02:09,  7.01it/s]2026-05-07 06:38:23.320 | WARNING  | gsloc.datasets.three_rscan:_warn_missing_asset:451 - Missing or unreadable graph /mnt/external_usb_hdd/6YL/Datasets/3RScan/SceneGraphs_Makarov_FULL_TEST_pt/42384908-60a7-271e-9c46-01e562c8974c/frame-000017.pt
2026-05-07 06:38:23.352 | WARNING  | gsloc.datasets.three_rscan:_warn_missing_asset:451 - Missing or unreadable graph /mnt/external_usb_hdd/6YL/Datasets/3RScan/SceneGraphs_Makarov_FULL_TEST_pt/42384908-60a7-271e-9c46-01e562c8974c/frame-000018.pt
Compute descriptors + PR cache: 100%|██████████| 1314/1314 [02:49<00:00,  7.75it/s]
2026-05-07 06:40:20.445 | INFO     | gsloc.datasets.pr_dataset:save_meta_parquet:51 - Wrote 21,013 rows to /home/kartashov_ga/projects/GSLoc/data/tests/26-05-06/makarov/64_graph_query_cache/meta.parquet
  0%|          | 0/11 [00:03<?, ?it/s]


KeyboardInterrupt: 

In [7]:
cfg.rerank_model = megaLoc
for i, similarity_kwargs in enumerate(similarity_kwargs_list):
    cfg.similarity_kwargs = similarity_kwargs
    cfg.bench_report_path = cfg.test_path  / similarity_names[i]
    cfg.frames_path = cfg.test_path / "frames.npz"
    test = Test(cfg)
    test.run()

2026-05-07 06:42:06.266 | INFO     | mmpr.inference.index:generate:439 - Using existing meta.parquet
2026-05-07 06:42:06.267 | INFO     | mmpr.inference.index:generate:468 - Using existing descriptors.npy
2026-05-07 06:42:06.268 | INFO     | mmpr.inference.index:generate:486 - schema.json file was saved in /home/kartashov_ga/projects/GSLoc/data/tests/26-05-06/makarov/64_graph_index
2026-05-07 06:42:06.297 | INFO     | mmpr.inference.index:generate:439 - Using existing meta.parquet
2026-05-07 06:42:06.298 | INFO     | mmpr.inference.index:generate:468 - Using existing descriptors.npy


2026-05-07 06:42:06.974 | INFO     | mmpr.inference.index:generate:486 - schema.json file was saved in /home/kartashov_ga/projects/GSLoc/data/tests/26-05-06/megaloc_index


Running inference and saving frames to /home/kartashov_ga/projects/GSLoc/data/tests/26-05-06/makarov/64xMegaloc/frames.npz
Using cached descriptors for query and rerank: loading from  /home/kartashov_ga/projects/GSLoc/data/tests/26-05-06/makarov/64_graph_query_cache /home/kartashov_ga/projects/GSLoc/data/tests/26-05-06/megaloc_query_cache
Descriptors loaded:  (21013, 64) (21013, 8448) getting results


retrieval: 21013it [01:13, 286.91it/s]


Results got:  21013


...

Index1 search time mean: 0.0
Rerank index2 search time mean: 0.0


2026-05-07 06:58:50.378 | INFO     | mmpr.inference.index:generate:486 - schema.json file was saved in /home/kartashov_ga/projects/GSLoc/data/tests/26-05-06/megaloc_index


Loading frames from /home/kartashov_ga/projects/GSLoc/data/tests/26-05-06/makarov/64xMegaloc/frames.npz


100%|██████████| 11/11 [09:31<00:00, 52.00s/it]

Index1 search time mean: 0.0
Rerank index2 search time mean: 0.0


In [10]:
plot_metrics_from_parquet(
    "/home/kartashov_ga/projects/GSLoc/data/tests/26-05-06/makarov/64xMegaloc/pose-near-sim/summaryresults.parquet",
    "/home/kartashov_ga/projects/GSLoc/data/tests/26-05-06/makarov/Megaloc/nearfilter_seq_report/pose-near-sim/summaryresults.parquet",
    metrics=["recall_at_1", "recall_at_5", "recall_at_10", "recall_at_25"]
)

{'recall_at_1': Figure({
     'data': [{'error_y': {'array': {'bdata': ('tV8J2kPsqj+3ez6hdPyrP0a2E3ezUK' ... 'aG1aU/2S9fuoHFpz92GlmZmLapPw=='),
                                     'dtype': 'f8'}},
               'hovertemplate': 'w=%{x}<br>recall_at_1=%{y}<extra></extra>',
               'legendgroup': '',
               'line': {'color': '#636efa', 'dash': 'solid'},
               'marker': {'symbol': 'circle'},
               'mode': 'lines+markers',
               'name': '',
               'orientation': 'v',
               'showlegend': False,
               'type': 'scatter',
               'x': {'bdata': 'AQIDBQcKDxQZHiM=', 'dtype': 'i1'},
               'xaxis': 'x',
               'y': {'bdata': ('vM0lnWlX1T92ZZhXwvDWP4/r1QXdGt' ... '5mktM/XnjIFfQJ0z+RWp7SmADTPw=='),
                     'dtype': 'f8'},
               'yaxis': 'y'},
              {'marker': {'color': 'red', 'size': 10},
               'mode': 'markers',
               'name': 'max',
               'showlegend